In [1]:
# !pip install -r requirements.txt
# !pip install tensorboard

In [2]:
import os
import numpy as np
import pandas as pd
import pickle as pkl

from model import Model
from data_handler import DataHandler

In [3]:
data_dir = 'Data'
os.makedirs(data_dir, exist_ok=True)
ticker = 'AAPL'
start_date = '2024-07-01'
end_date = '2024-08-30'
data_handler = DataHandler(data_dir)

In [ ]:
processed_data = data_handler.get_data(ticker, start_date, end_date, split_train_test=True)
processed_train_data, processed_test_data = processed_data

In [ ]:
processed_train_data.describe()

In [6]:
model_dir = 'Models'
# os.makedirs(model_dir, exist_ok=True)
logging_dir = 'logs'

trial = "_env6_trial3_totpen_multi_env_exp_rew_price_sde_false_allinv_buy"

n_env = 1

sac_model = Model(model_dir=model_dir, logging_dir=logging_dir,n_env=n_env)

training_params = {
    "total_timesteps": 100000,
    "callback" : None
}

# training_config = {
#     "learning_rate": 1e-3,
#     "n_steps": 64,
#     "batch_size": 256,
#     "gamma": 0.9,
#     "clip_range" : 0.2,
#     "n_epochs": 4,
#     "ent_coef": 0.01,
# }

training_config = None

env_params = {
    "preferred_timeframe" : 390,
    "inventory": 10000,
    "action" : "buy"
}

model_name = f"{ticker}_trial{trial}.pt"

training_params['tb_log_name'] =f'{ticker}_trial{trial}'




In [ ]:
trained_model = sac_model.train(processed_train_data, resume=False, model_name=model_name, env_params=env_params, training_config=training_config, training_params=training_params)

In [ ]:
env_params["preferred_timeframe"] = 390
rew, trades = lstm_ppo_model.test(processed_test_data, model=None, model_name=model_name, env_params=env_params)
print(" Reward: ", rew)
trades.to_csv(f"{ticker}_trades.csv")
trades

In [ ]:
10000/390

# Fine Tuning

In [10]:
# from ray import train, tune
# from ray.tune.schedulers import ASHAScheduler
# from ray.tune.search.hyperopt import HyperOptSearch

In [11]:
# # from ray import tune, train
# # from ray.tune.schedulers import ASHAScheduler
# # from ray.tune.search.hyperopt import HyperOptSearch
# # import ray

# model_dir = os.path.abspath('./Models_tuning')
# # os.makedirs(model_dir, exist_ok=True)
# print(model_dir)
# # No logging
# lstm_ppo_model = Model(model_dir=model_dir, stats_window_size=1000, logging_dir=None)

# # Training Config
# """
# Key required for fine tuning
# learning_rate, n_steps, batch_size, gamma, clip_range, n_epochs, ent_coef, resume, total_timesteps, callback, tb_log_name, train_datatest_data
# """

# training_params = {
#     "total_timesteps": 5000,
#     "callback" : None
# }

# finetune_config = {
#     "learning_rate": tune.loguniform(1e-5, 1e-1),
#     "n_steps": tune.choice([32, 64, 128, 256, 512]),
#     "batch_size": tune.choice([64, 128, 256]),
#     "gamma": tune.uniform(0.9, 0.999),
#     "clip_range" : tune.uniform(0.1, 0.4),
#     "n_epochs": tune.choice([4, 6, 8]),
#     "vf_coef" : tune.loguniform(0.1, 0.8),

#     "ent_coef": tune.loguniform(0.0001, 0.1),
#     "resume" : False,
#     "total_timesteps": training_params['total_timesteps'],
#     "callback" : training_params['callback'],
#     "env_params": {
#         "preferred_timeframe" : 390,
#         "inventory": 10000
#     }
# }

# ray.init(num_gpus=1)

# scheduler = ASHAScheduler()

# hyperopt_search = HyperOptSearch()


# finetune_config["train_data"] = processed_train_data
# finetune_config["test_data"] = processed_test_data

# results_dir = os.path.abspath("./tune_results_tmp")

# analysis = tune.Tuner(
#     lstm_ppo_model.fine_tuning_model,
#     param_space=finetune_config,
#     run_config=train.RunConfig(
#         name=f"{ticker}",
#         storage_path=results_dir,
#     ),
#     tune_config=tune.TuneConfig(
#         search_alg=hyperopt_search,
#         scheduler=scheduler,
#         num_samples=1,
#         trial_dirname_creator=lambda trial: trial.trial_id,
#         metric="reward",
#         mode="max",),
# )

# results = analysis.fit()
# print(results)
# results.get_dataframe().to_csv(f"{results_dir}/tune_results.csv")
# best_results = results.get_best_result(metric="reward", mode="max")
# # best_config = analysis.get_best_config(metric="reward", mode="max")

# print("Best config:", best_results.config)
# best_config = best_results.config
# # Save best parameters
# # save_dir = "./best_ppo_params"
# # os.makedirs(save_dir, exist_ok=True)
# del best_config["train_data"]
# del best_config["test_data"]
# pkl.dump(best_config, open(os.path.join(results_dir, "best_ppo_params.pkl"), "wb"))
# with open(os.path.join(results_dir, "best_config.txt"), "w") as f:
#     for key, value in best_config.items():
#         f.write(f"{key}: {value}\n")
# ray.shutdown()
# # print("Best config: ", analysis.get_best_config(metric="mean_reward", mode="max"))
# # df = analysis.results_df
# # print(df.head())

In [12]:
# !pip install -U stable-baselines3
# !pip install -U hyperopt